In [ ]:
pip install torch
python ejercicio3_minillm_transformer.py texto.txt

In [ ]:
import math
import sys
from dataclasses import dataclass
from typing import List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# -----------------------------
# Datos: texto -> ids
# -----------------------------
def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    # limpieza mínima opcional
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text

def build_vocab(text: str):
    chars = sorted(list(set(text)))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos

def encode(text: str, stoi: dict) -> torch.Tensor:
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

def decode(ids: List[int], itos: dict) -> str:
    return "".join(itos[i] for i in ids)


# -----------------------------
# Dataset: ventanas de longitud L
# -----------------------------
class CharWindowDataset(Dataset):
    def __init__(self, data_ids: torch.Tensor, seq_len: int):
        self.data = data_ids
        self.seq_len = seq_len

    def __len__(self):
        return max(0, len(self.data) - self.seq_len - 1)

    def __getitem__(self, idx: int):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y


# -----------------------------
# Bloques Transformer causal
# -----------------------------
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

        # la máscara causal se construye dinámicamente por longitud T

    def forward(self, x):
        # x: (B, T, C)
        B, T, C = x.size()
        qkv = self.qkv(x)  # (B, T, 3C)
        q, k, v = qkv.split(C, dim=2)

        # reshape a (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # atención: (B, n_head, T, T)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # máscara causal (triangular)
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, 1, T, T)
        att = att.masked_fill(mask == 0, float("-inf"))

        att = torch.softmax(att, dim=-1)
        att = self.dropout(att)

        out = att @ v  # (B, n_head, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)
        out = self.proj(out)
        out = self.dropout(out)
        return out


class MLP(nn.Module):
    def __init__(self, n_embd: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(n_embd, 4 * n_embd)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(self.act(self.fc1(x))))


class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


# -----------------------------
# Mini-Transformer LM
# -----------------------------
class MiniTransformerLM(nn.Module):
    def __init__(self, vocab_size: int, seq_len: int, n_embd: int = 256, n_head: int = 4, n_layer: int = 4, dropout: float = 0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.seq_len = seq_len

        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(seq_len, n_embd)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx):
        # idx: (B, T)
        B, T = idx.size()
        assert T <= self.seq_len, "T excede seq_len del modelo"

        tok = self.tok_emb(idx)  # (B, T, C)
        pos = self.pos_emb(torch.arange(T, device=idx.device))  # (T, C)
        x = self.drop(tok + pos.unsqueeze(0))  # (B, T, C)

        for blk in self.blocks:
            x = blk(x)

        x = self.ln_f(x)
        logits = self.head(x)  # (B, T, V)
        return logits


# -----------------------------
# Entrenamiento y evaluación
# -----------------------------
@dataclass
class TrainConfig:
    seq_len: int = 128
    batch_size: int = 64
    n_embd: int = 256
    n_head: int = 4
    n_layer: int = 4
    dropout: float = 0.1
    lr: float = 3e-4
    epochs: int = 5
    device: str = "cpu"
    temperature_list: Tuple[float, ...] = (0.7, 1.0, 1.2)
    gen_tokens: int = 400
    prompt: str = "La inteligencia artificial "


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    criterion = nn.CrossEntropyLoss()
    total = 0.0
    n = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        logits = model(x)  # (B,T,V)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
        n += 1
    return total / max(1, n)

@torch.no_grad()
def eval_loss(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total = 0.0
    n = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        total += loss.item()
        n += 1
    return total / max(1, n)

@torch.no_grad()
def generate(model, prompt: str, stoi: dict, itos: dict, seq_len: int, max_new_tokens: int, temperature: float, device: str):
    model.eval()
    device = torch.device(device)

    ids = [stoi[ch] for ch in prompt if ch in stoi]
    if not ids:
        ids = [0]

    out = ids[:]
    for _ in range(max_new_tokens):
        # recortar a ventana máxima
        ctx = out[-seq_len:]
        x = torch.tensor(ctx, dtype=torch.long, device=device).unsqueeze(0)  # (1,T)
        logits = model(x)  # (1,T,V)
        logits = logits[0, -1, :]  # (V,)

        logits = logits / max(1e-8, temperature)
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()
        out.append(next_id)

    return decode(out, itos)


def main(text_path: str, cfg: TrainConfig):
    text = load_text(text_path)
    stoi, itos = build_vocab(text)
    data_ids = encode(text, stoi)

    # split train/val
    n = len(data_ids)
    split = int(0.9 * n)
    train_ids = data_ids[:split]
    val_ids = data_ids[split:]

    train_ds = CharWindowDataset(train_ids, cfg.seq_len)
    val_ds = CharWindowDataset(val_ids, cfg.seq_len)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=True)

    device = torch.device(cfg.device)

    model = MiniTransformerLM(
        vocab_size=len(stoi),
        seq_len=cfg.seq_len,
        n_embd=cfg.n_embd,
        n_head=cfg.n_head,
        n_layer=cfg.n_layer,
        dropout=cfg.dropout
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)

    print(f"Vocab size: {len(stoi)} | Train windows: {len(train_ds)} | Val windows: {len(val_ds)} | Device: {cfg.device}")
    print(f"Model: layers={cfg.n_layer}, heads={cfg.n_head}, embd={cfg.n_embd}, seq_len={cfg.seq_len}")

    for epoch in range(1, cfg.epochs + 1):
        tr = train_one_epoch(model, train_loader, optimizer, device)
        vl = eval_loss(model, val_loader, device)
        ppl = math.exp(vl) if vl < 20 else float("inf")
        print(f"Epoch {epoch:02d} | train_loss={tr:.4f} | val_loss={vl:.4f} | val_ppl={ppl:.2f}")

    # Generación
    for temp in cfg.temperature_list:
        sample = generate(
            model,
            cfg.prompt,
            stoi,
            itos,
            seq_len=cfg.seq_len,
            max_new_tokens=cfg.gen_tokens,
            temperature=temp,
            device=cfg.device
        )
        print("\n" + "=" * 70)
        print(f"Temperature = {temp}")
        print(sample[:2000])

In [ ]:
if __name__ == "__main__":
    #Si es CPU y tarda:
    # n_layer=2, n_embd=128, batch_size=32, epochs=3.
    #Si usan GPU:
    # mantén n_layer=4, n_embd=256, epochs=5–10.
    cfg = TrainConfig(
        seq_len=128,
        batch_size=64,
        n_embd=256,
        n_head=4,
        n_layer=4,
        dropout=0.1,
        lr=3e-4,
        epochs=5,
        device="cpu",  # cambia a "cuda" si tienes GPU
        temperature_list=(0.7, 1.0, 1.2),
        gen_tokens=400,
        prompt="La inteligencia artificial "
    )
    main("texto.txt", cfg)